# Data Exploration + Mini Training/Eval Simulation

Notebook operativo per capire la pipeline della repo senza fare il training reale da cluster.

Obiettivi:
- esplorare struttura dati e clip
- leggere e confrontare le config (teacher, baseline, distillation, attention transfer)
- simulare mini training + mini evaluation con pochi batch
- osservare differenze di flusso tra le modalita

Nota importante: questa e una simulazione didattica, non una run comparabile ai risultati ufficiali su cluster.

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Subset

# Ensure repo root is importable when notebook is opened from notebooks/
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.datasets.ucf101 import get_dataloaders
from src.evaluation.metrics import compute_inference_time, compute_model_size
from src.models.student import get_student
from src.models.teacher import get_teacher
from src.training.losses import CombinedKDATLoss, KDLoss
from src.utils.config import load_config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Repo root: {repo_root}")

In [ ]:
# Config files used by the official training pipeline
config_paths = {
    "teacher": repo_root / "experiments/configs/teacher.yaml",
    "baseline": repo_root / "experiments/configs/baseline.yaml",
    "distillation": repo_root / "experiments/configs/distillation.yaml",
    "attention_transfer": repo_root / "experiments/configs/attention_transfer.yaml",
}

raw_configs = {name: load_config(str(path)) for name, path in config_paths.items()}

summary_rows = []
for name, cfg in raw_configs.items():
    summary_rows.append({
        "config": name,
        "mode": cfg["training"]["mode"],
        "model_type": cfg["model"]["type"],
        "epochs": cfg["training"]["epochs"],
        "batch_size": cfg["training"]["batch_size"],
        "lr": cfg["training"]["lr"],
        "num_frames": cfg["dataset"]["num_frames"],
        "crop_size": cfg["dataset"]["crop_size"],
        "backend": cfg["dataset"].get("backend", "n/a"),
        "uses_teacher_ckpt": "distillation" in cfg and "teacher_checkpoint" in cfg.get("distillation", {}),
    })

pd.DataFrame(summary_rows).sort_values("config")

## Mini-run settings

Riduciamo tutto per velocita:
- batch molto piccolo
- pochi frame
- pochi batch train/eval
- teacher senza pesi pretrained (se non disponibili localmente)

Se il dataset non e disponibile, useremo un dataset sintetico con stesso shape [C, T, H, W].

In [ ]:
class SyntheticVideoDataset(Dataset):
    def __init__(self, n_samples=64, num_classes=101, num_frames=8, crop_size=96, seed=42):
        self.n_samples = n_samples
        self.num_classes = num_classes
        self.num_frames = num_frames
        self.crop_size = crop_size
        rng = np.random.default_rng(seed)
        self.labels = rng.integers(0, num_classes, size=n_samples).tolist()

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        clip = torch.randn(3, self.num_frames, self.crop_size, self.crop_size)
        label = int(self.labels[idx])
        return clip, label


def make_tiny_config(cfg):
    c = copy.deepcopy(cfg)
    c.setdefault("logging", {})["enabled"] = False

    c["dataset"]["num_frames"] = 8
    c["dataset"]["crop_size"] = 96

    c["training"]["batch_size"] = 2
    c["training"]["num_workers"] = 0
    c["training"]["mixed_precision"] = device.type == "cuda"

    return c


def get_tiny_dataloaders(cfg, n_train=16, n_test=8):
    tiny_cfg = make_tiny_config(cfg)

    try:
        dls = get_dataloaders(tiny_cfg)

        train_ds = dls["train"].dataset
        test_ds = dls["test"].dataset

        train_n = min(n_train, len(train_ds))
        test_n = min(n_test, len(test_ds))

        train_idx = list(range(train_n))
        test_idx = list(range(test_n))

        train_loader = DataLoader(
            Subset(train_ds, train_idx),
            batch_size=tiny_cfg["training"]["batch_size"],
            shuffle=True,
            num_workers=0,
            drop_last=True,
        )
        test_loader = DataLoader(
            Subset(test_ds, test_idx),
            batch_size=tiny_cfg["training"]["batch_size"],
            shuffle=False,
            num_workers=0,
        )

        source = "real"
    except Exception as exc:
        print(f"Fallback to synthetic dataset: {exc}")
        num_classes = tiny_cfg["dataset"].get("num_classes", 101)
        num_frames = tiny_cfg["dataset"].get("num_frames", 8)
        crop_size = tiny_cfg["dataset"].get("crop_size", 96)

        train_loader = DataLoader(
            SyntheticVideoDataset(
                n_samples=n_train,
                num_classes=num_classes,
                num_frames=num_frames,
                crop_size=crop_size,
            ),
            batch_size=tiny_cfg["training"]["batch_size"],
            shuffle=True,
            num_workers=0,
            drop_last=True,
        )
        test_loader = DataLoader(
            SyntheticVideoDataset(
                n_samples=n_test,
                num_classes=num_classes,
                num_frames=num_frames,
                crop_size=crop_size,
                seed=123,
            ),
            batch_size=tiny_cfg["training"]["batch_size"],
            shuffle=False,
            num_workers=0,
        )
        source = "synthetic"

    return tiny_cfg, {"train": train_loader, "test": test_loader}, source

In [ ]:
def build_models_and_loss(cfg, device):
    mode = cfg["training"]["mode"]
    num_classes = cfg["dataset"].get("num_classes", 101)

    teacher = None

    if mode == "teacher_finetune":
        model = get_teacher(
            num_classes=num_classes,
            pretrained=False,
            freeze_backbone=cfg["model"].get("freeze_backbone", False),
            extract_features=False,
        )
        criterion = torch.nn.CrossEntropyLoss()

    elif mode == "baseline":
        model = get_student(
            num_classes=num_classes,
            width_mult=cfg["model"].get("width_mult", 1.0),
            extract_features=False,
        )
        criterion = torch.nn.CrossEntropyLoss()

    elif mode == "distillation":
        model = get_student(
            num_classes=num_classes,
            width_mult=cfg["model"].get("width_mult", 1.0),
            extract_features=False,
        )
        teacher = get_teacher(
            num_classes=num_classes,
            pretrained=False,
            extract_features=False,
        )
        teacher.eval()
        criterion = KDLoss(
            temperature=cfg["distillation"].get("temperature", 5.0),
            alpha=cfg["distillation"].get("alpha", 0.7),
        )

    elif mode == "distillation_at":
        model = get_student(
            num_classes=num_classes,
            width_mult=cfg["model"].get("width_mult", 1.0),
            extract_features=True,
        )
        teacher = get_teacher(
            num_classes=num_classes,
            pretrained=False,
            extract_features=True,
        )
        teacher.eval()
        criterion = CombinedKDATLoss(
            temperature=cfg["distillation"].get("temperature", 5.0),
            alpha=cfg["distillation"].get("alpha", 0.7),
            beta=cfg["distillation"].get("at_beta", 0.1),
            teacher_keys=cfg["distillation"].get("teacher_keys", [3, 4, 5]),
            student_keys=cfg["distillation"].get("student_keys", [2, 4, 6]),
        )

    else:
        raise ValueError(f"Unsupported mode: {mode}")

    model = model.to(device)
    if teacher is not None:
        teacher = teacher.to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=cfg["training"].get("lr", 0.01),
        momentum=cfg["training"].get("momentum", 0.9),
        weight_decay=cfg["training"].get("weight_decay", 1e-4),
    )

    return model, teacher, criterion, optimizer

In [ ]:
def mini_train_eval(cfg, loaders, device, max_train_steps=3, max_eval_steps=2):
    mode = cfg["training"]["mode"]
    model, teacher, criterion, optimizer = build_models_and_loss(cfg, device)

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    at_feature_shapes = None

    t0 = time.perf_counter()
    for step, (clips, labels) in enumerate(loaders["train"]):
        if step >= max_train_steps:
            break

        clips = clips.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        if mode in ("teacher_finetune", "baseline"):
            logits = model(clips)
            loss = criterion(logits, labels)

        elif mode == "distillation":
            logits = model(clips)
            with torch.no_grad():
                t_logits = teacher(clips)
            loss = criterion(logits, t_logits, labels)

        elif mode == "distillation_at":
            logits = model(clips)
            with torch.no_grad():
                t_logits = teacher(clips)
            t_feats = teacher.get_intermediate_features()
            s_feats = model.get_intermediate_features()
            total, kd_comp, at_comp = criterion(logits, t_logits, labels, t_feats, s_feats)
            loss = total
            at_feature_shapes = {
                "teacher": {k: tuple(v.shape) for k, v in t_feats.items()},
                "student": {k: tuple(v.shape) for k, v in s_feats.items()},
            }

        loss.backward()
        optimizer.step()

        train_loss += float(loss.item()) * labels.size(0)
        preds = logits.argmax(dim=1)
        train_correct += int((preds == labels).sum().item())
        train_total += labels.size(0)

    train_time_s = time.perf_counter() - t0

    model.eval()
    eval_correct1 = 0
    eval_correct5 = 0
    eval_total = 0

    with torch.no_grad():
        for step, (clips, labels) in enumerate(loaders["test"]):
            if step >= max_eval_steps:
                break

            clips = clips.to(device)
            labels = labels.to(device)
            logits = model(clips)

            pred1 = logits.argmax(dim=1)
            eval_correct1 += int((pred1 == labels).sum().item())

            topk = min(5, logits.shape[1])
            _, pred5 = logits.topk(topk, dim=1)
            eval_correct5 += int((pred5 == labels.unsqueeze(1)).any(dim=1).sum().item())

            eval_total += labels.size(0)

    train_loss = train_loss / max(train_total, 1)
    train_acc = 100.0 * train_correct / max(train_total, 1)
    eval_top1 = 100.0 * eval_correct1 / max(eval_total, 1)
    eval_top5 = 100.0 * eval_correct5 / max(eval_total, 1)

    size_info = compute_model_size(model)
    in_shape = (
        3,
        cfg["dataset"]["num_frames"],
        cfg["dataset"]["crop_size"],
        cfg["dataset"]["crop_size"],
    )
    timing = compute_inference_time(
        model,
        in_shape,
        device,
        num_runs=10,
        warmup_runs=3,
    )

    return {
        "mode": mode,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "eval_top1": eval_top1,
        "eval_top5": eval_top5,
        "param_count": size_info["param_count"],
        "model_size_mb": size_info["size_mb"],
        "inference_ms": timing["avg_ms"],
        "train_time_s": train_time_s,
        "at_feature_shapes": at_feature_shapes,
    }

In [ ]:
# Optional clip visualization from the first available config
tiny_cfg, tiny_loaders, source = get_tiny_dataloaders(raw_configs["baseline"])
clips, labels = next(iter(tiny_loaders["train"]))
print(f"Data source: {source}")
print(f"Batch clip shape: {tuple(clips.shape)} | labels shape: {tuple(labels.shape)}")

# clips: [B, C, T, H, W]
first_clip = clips[0].permute(1, 2, 3, 0).numpy()
# Normalize for visualization
first_clip_vis = (first_clip - first_clip.min()) / (first_clip.max() - first_clip.min() + 1e-8)

n_show = min(6, first_clip_vis.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
for i in range(n_show):
    axes[i].imshow(first_clip_vis[i])
    axes[i].set_title(f"t={i}")
    axes[i].axis("off")
plt.suptitle("Sample temporal frames from one clip")
plt.tight_layout()
plt.show()

In [ ]:
results = []

for cfg_name, base_cfg in raw_configs.items():
    print(f"\nRunning mini simulation for: {cfg_name}")
    tiny_cfg, tiny_loaders, source = get_tiny_dataloaders(base_cfg, n_train=16, n_test=8)

    out = mini_train_eval(
        tiny_cfg,
        tiny_loaders,
        device=device,
        max_train_steps=3,
        max_eval_steps=2,
    )
    out["config"] = cfg_name
    out["data_source"] = source
    results.append(out)

res_df = pd.DataFrame(results)
display_cols = [
    "config",
    "mode",
    "data_source",
    "train_loss",
    "train_acc",
    "eval_top1",
    "eval_top5",
    "model_size_mb",
    "inference_ms",
    "train_time_s",
]
res_df[display_cols].sort_values("config")

In [ ]:
# Visual comparison of size, speed, and mini-accuracy
plot_df = res_df.sort_values("config").reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(plot_df["config"], plot_df["model_size_mb"])
axes[0].set_title("Model size (MB)")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(plot_df["config"], plot_df["inference_ms"])
axes[1].set_title("Inference time (ms, mini benchmark)")
axes[1].tick_params(axis="x", rotation=25)

axes[2].bar(plot_df["config"], plot_df["eval_top1"])
axes[2].set_title("Mini eval Top-1 (%)")
axes[2].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

## Lettura finale

Come interpretare i risultati di questo notebook:
- metriche e accuracy qui servono per capire il flusso del codice, non per confronto scientifico
- il confronto size/speed e utile per visualizzare il trade-off teacher vs student
- le modalita `distillation` e `distillation_at` mostrano cosa cambia nella loss e nelle feature intermedie
- il training reale resta quello su cluster (`cluster/train.sh`, config ufficiali, dataset completo, job lunghi)

Per una run piu vicina alla realta puoi aumentare gradualmente:
- `n_train`, `n_test` nei loader ridotti
- `max_train_steps` e `max_eval_steps`
- `num_frames` e `crop_size`

Quando passi al cluster, mantieni questi notebook come strumento di debug rapido e ispezione pipeline.